In [1]:
import tkinter as tk from tkinter import ttk, messagebox from tkinter.scrolledtext import ScrolledText import pandas as pd import random import os import threading from deep_translator import GoogleTranslator # ---------------- CONFIGURATION ---------------APP_TITLE = "WordGarden | Language & Vocabulary Studio" FILE_NAME = "wordgarden_favourites.csv" BG = "#FFF8F5" PINK = "#F5D9E5" DARK_PINK = "#AD527B" PURPLE = "#8571AA" DARK = "#46394F" WHITE = "#FFFFFF" GREEN = "#DDEBDD" LANGUAGES = {    "English": "en", "Hindi": "hi", "French": "fr",    "Spanish": "es", "German": "de", "Italian": "it",    "Japanese": "ja", "Korean": "ko", "Chinese": "zh-CN",    "Arabic": "ar", "Punjabi": "pa", "Bengali": "bn",    "Urdu": "ur", "Portuguese": "pt", "Russian": "ru", } WORDS = [ Page 2 WordGarden | Python Project    ("Serendipity", "Finding something beautiful by chance."),    ("Bloom", "To grow, flourish, or develop beautifully."),    ("Resilience", "The ability to recover after difficulties."),    ("Gratitude", "The feeling of being thankful."),    ("Tranquility", "A state of peace and calm."),    ("Wonder", "A feeling of surprise and admiration."),    ("Kindness", "The quality of being caring and helpful."),    ("Harmony", "A pleasing balance between different things."),    ("Courage", "The ability to face fear or difficulty."),    ("Radiant", "Shining brightly with happiness or light."), ] # ---------------- DATA MANAGEMENT ---------------def load_favourites():    """Load saved vocabulary from a CSV file."""    if os.path.exists(FILE_NAME):        try:            return pd.read_csv(FILE_NAME).fillna("")        except (pd.errors.EmptyDataError, pd.errors.ParserError):            pass    return pd.DataFrame(columns=[        "Original Text", "Translation",        "Source Language", "Target Language"    ]) def save_favourite():    """Save the current translation to the favourites file."""    original = source_text.get("1.0", tk.END).strip()    translated = result_text.get("1.0", tk.END).strip()    if not original or not translated:        messagebox.showwarning(            "Nothing to save",            "Please translate some text first."        )        return Page 3 WordGarden | Python Project

SyntaxError: cannot use name as import target (99600254.py, line 1)

In [1]:
import os
import random
import threading
import tkinter as tk
from tkinter import messagebox, ttk
from tkinter.scrolledtext import ScrolledText

import pandas as pd
from deep_translator import MyMemoryTranslator

# ============================================================
# WORDGARDEN — EDITORIAL CHIC
# Tkinter + MyMemoryTranslator. Internet connection required.
# ============================================================
APP_TITLE = "WordGarden | Editorial Chic"
FILE_NAME = "wordgarden_favourites.csv"

LANGUAGES = {
    "English": "en-GB",
    "Hindi": "hi-IN",
    "French": "fr-FR",
    "Spanish": "es-ES",
    "German": "de-DE",
    "Italian": "it-IT",
    "Japanese": "ja-JP",
    "Korean": "ko-KR",
    "Chinese": "zh-CN",
    "Arabic": "ar-SA",
    "Punjabi": "pa-IN",
    "Bengali": "bn-IN",
    "Urdu": "ur-PK",
    "Portuguese": "pt-PT",
}

WORDS = [
    ("Serendipity", "Finding something beautiful by chance."),
    ("Bloom", "To grow, flourish, or develop beautifully."),
    ("Resilience", "The ability to recover after difficulties."),
    ("Gratitude", "The feeling of being thankful."),
    ("Tranquility", "A state of peace and calm."),
    ("Wonder", "A feeling of surprise and admiration."),
    ("Kindness", "The quality of being caring and helpful."),
    ("Harmony", "A pleasing balance between different things."),
    ("Courage", "The ability to face fear or difficulty."),
    ("Radiant", "Shining brightly with happiness or light."),
]

FAVOURITE_COLUMNS = ["Original Text", "Translation", "Source Language", "Target Language"]

THEMES = {
    "light": {
        "bg": "#FAF7F2", "surface": "#FFFFFF", "surface_alt": "#F1EAE2",
        "text": "#242222", "muted": "#766B67", "accent": "#873C4A",
        "accent_hover": "#6F2F3B", "accent_text": "#FFFFFF", "line": "#E2D6CC",
        "input": "#FFFEFC", "output": "#F5EEE8", "gold": "#B58B52",
        "button_secondary": "#E9DFD5", "button_secondary_text": "#332B29",
    },
    "dark": {
        "bg": "#191719", "surface": "#252124", "surface_alt": "#30292D",
        "text": "#F7EFE9", "muted": "#C0B0AA", "accent": "#D27A89",
        "accent_hover": "#B85E70", "accent_text": "#201719", "line": "#493B40",
        "input": "#211E20", "output": "#2D2529", "gold": "#E2BD7B",
        "button_secondary": "#40363A", "button_secondary_text": "#F7EFE9",
    },
}

current_theme = "light"
translation_busy = False
animation_phase = 0
spinner_phase = 0
# Works both when launched as a .py file and when run in Jupyter/IPython,
# where __file__ is not defined. In notebooks, use the current working folder.
if "__file__" in globals():
    APP_DIRECTORY = os.path.dirname(os.path.abspath(__file__))
else:
    APP_DIRECTORY = os.getcwd()
FAVOURITE_FILE = os.path.join(APP_DIRECTORY, FILE_NAME)


def load_favourites():
    """Load favourites while keeping older CSV files compatible."""
    if os.path.exists(FAVOURITE_FILE):
        try:
            df = pd.read_csv(FAVOURITE_FILE).fillna("")
            for column in FAVOURITE_COLUMNS:
                if column not in df.columns:
                    df[column] = ""
            return df[FAVOURITE_COLUMNS]
        except (pd.errors.EmptyDataError, pd.errors.ParserError, OSError):
            pass
    return pd.DataFrame(columns=FAVOURITE_COLUMNS)


def save_favourite():
    original = source_text.get("1.0", tk.END).strip()
    translated = result_text.get("1.0", tk.END).strip()
    if not original or not translated or translated in {"Translating…", "Translating..."}:
        messagebox.showwarning("Nothing to save", "Enter text and complete a translation first.", parent=root)
        return
    row = pd.DataFrame([{
        "Original Text": original,
        "Translation": translated,
        "Source Language": source_lang_combo.get(),
        "Target Language": target_lang_combo.get(),
    }])
    try:
        df = pd.concat([load_favourites(), row], ignore_index=True)
        df.to_csv(FAVOURITE_FILE, index=False, encoding="utf-8-sig")
        status_var.set("Saved to your vocabulary collection ✦")
        messagebox.showinfo("Added to your collection", "Your translation has been saved to favourites.", parent=root)
    except OSError as exc:
        messagebox.showerror("Could not save", f"Unable to write the favourites file:\n{exc}", parent=root)


def show_favourites():
    df = load_favourites()
    window = tk.Toplevel(root)
    window.title("Your WordGarden Collection")
    window.geometry("820x420")
    window.minsize(600, 300)
    t = THEMES[current_theme]
    window.configure(bg=t["bg"])
    tk.Label(window, text="YOUR WORDGARDEN COLLECTION", font=("Georgia", 16, "bold"),
             bg=t["bg"], fg=t["accent"]).pack(anchor="w", padx=20, pady=(18, 4))
    tk.Label(window, text="A little archive of words worth keeping.", font=("Helvetica", 10),
             bg=t["bg"], fg=t["muted"]).pack(anchor="w", padx=20, pady=(0, 12))
    if df.empty:
        tk.Label(window, text="Your collection is waiting for its first word ✦", font=("Georgia", 12),
                 bg=t["bg"], fg=t["text"]).pack(pady=35)
        return
    frame = tk.Frame(window, bg=t["bg"])
    frame.pack(fill=tk.BOTH, expand=True, padx=16, pady=(0, 16))
    columns = tuple(FAVOURITE_COLUMNS)
    tree = ttk.Treeview(frame, columns=columns, show="headings")
    widths = (210, 280, 130, 130)
    for col, width in zip(columns, widths):
        tree.heading(col, text=col.upper())
        tree.column(col, width=width, minwidth=80, anchor="w")
    sy = ttk.Scrollbar(frame, orient="vertical", command=tree.yview)
    sx = ttk.Scrollbar(frame, orient="horizontal", command=tree.xview)
    tree.configure(yscrollcommand=sy.set, xscrollcommand=sx.set)
    tree.grid(row=0, column=0, sticky="nsew")
    sy.grid(row=0, column=1, sticky="ns")
    sx.grid(row=1, column=0, sticky="ew")
    frame.rowconfigure(0, weight=1)
    frame.columnconfigure(0, weight=1)
    for _, item in df.iterrows():
        tree.insert("", tk.END, values=[item.get(col, "") for col in columns])
    style.configure("Treeview", background=t["surface"], fieldbackground=t["surface"], foreground=t["text"])
    style.configure("Treeview.Heading", background=t["surface_alt"], foreground=t["text"])


def translate_text():
    global translation_busy
    if translation_busy:
        return
    text_to_translate = source_text.get("1.0", tk.END).strip()
    if not text_to_translate:
        messagebox.showwarning("A little note", "Write a word or sentence first, then translate it.", parent=root)
        return
    source_name = source_lang_combo.get().strip()
    target_name = target_lang_combo.get().strip()
    source_code = LANGUAGES.get(source_name)
    target_code = LANGUAGES.get(target_name)
    if not source_code or not target_code:
        messagebox.showerror("Language selection", "Please select valid source and target languages.", parent=root)
        return
    if source_code == target_code:
        messagebox.showwarning("Same language", "Choose two different languages to translate.", parent=root)
        return

    translation_busy = True
    result_text.configure(state=tk.NORMAL)
    result_text.delete("1.0", tk.END)
    result_text.insert("1.0", "Translating… ✦")
    btn_translate.configure(state=tk.DISABLED)
    status_var.set("Preparing your translation…")
    animate_spinner()

    def do_translation():
        try:
            translator = MyMemoryTranslator(source=source_code, target=target_code)
            translated = translator.translate(text_to_translate)
            if not translated:
                raise ValueError("The translation service returned an empty result.")
            root.after(0, show_translation_result, translated, None)
        except Exception as exc:
            root.after(0, show_translation_result, None, str(exc))

    threading.Thread(target=do_translation, daemon=True).start()


def show_translation_result(translated, error):
    global translation_busy
    translation_busy = False
    btn_translate.configure(state=tk.NORMAL)
    result_text.configure(state=tk.NORMAL)
    if error:
        result_text.delete("1.0", tk.END)
        status_var.set("Translation could not be completed. Please try again.")
        low = error.lower()
        if any(term in low for term in ("too many requests", "rate limit", "quota", "429")):
            message = ("The translation service is temporarily limiting requests. Wait before retrying.\n\nDetails: " + error)
        elif "no support for the provided language" in low or "unsupported" in low:
            message = ("The translation service did not accept one of these language codes. Try another language pair.\n\nDetails: " + error)
        else:
            message = ("Check your internet connection and selected languages, then try again.\n\nDetails: " + error)
        messagebox.showerror("Translation Error", message, parent=root)
        return
    result_text.delete("1.0", tk.END)
    result_text.insert("1.0", translated)
    status_var.set("Translation complete — beautifully said. ✦")


def load_random_word():
    word, definition = random.choice(WORDS)
    source_text.delete("1.0", tk.END)
    source_text.insert("1.0", f"{word}: {definition}")
    source_lang_combo.set("English")
    status_var.set("A word inspiration has arrived ✦")
    source_text.focus_set()


def clear_text():
    source_text.delete("1.0", tk.END)
    result_text.configure(state=tk.NORMAL)
    result_text.delete("1.0", tk.END)
    status_var.set("A fresh page awaits your words.")


def apply_theme():
    t = THEMES[current_theme]
    root.configure(bg=t["bg"])
    for frame in (topbar, hero, status_frame):
        frame.configure(bg=t["bg"])
    for frame in (controls_card, workspace_card, actions_frame, lang_inner, workspace):
        frame.configure(bg=t["surface"], highlightbackground=t["line"])
    for widget in (eyebrow, title_label, subtitle_label, section_from, section_to, arrow_label, input_label, output_label, status_label, footer_label):
        widget.configure(bg=t["bg"] if widget in (eyebrow, title_label, subtitle_label, status_label, footer_label) else t["surface"],
                         fg=t["accent"] if widget in (eyebrow, title_label, arrow_label) else t["muted"] if widget in (subtitle_label, footer_label) else t["text"])
    for widget in (source_text, result_text):
        widget.configure(bg=t["input"] if widget is source_text else t["output"], fg=t["text"],
                         insertbackground=t["text"], selectbackground=t["accent"], selectforeground=t["accent_text"])
    for combo in (source_lang_combo, target_lang_combo):
        combo.configure(background=t["surface_alt"], foreground=t["text"])
    theme_button.configure(bg=t["surface_alt"], fg=t["text"], activebackground=t["line"])
    for button in (btn_random, btn_save, btn_favourites, btn_clear):
        button.configure(bg=t["button_secondary"], fg=t["button_secondary_text"], activebackground=t["line"], activeforeground=t["text"])
    btn_translate.configure(bg=t["accent"], fg=t["accent_text"], activebackground=t["accent_hover"], activeforeground=t["accent_text"])
    style.configure("TCombobox", fieldbackground=t["surface_alt"], background=t["surface_alt"], foreground=t["text"], arrowcolor=t["accent"], padding=8)
    style.map("TCombobox", fieldbackground=[("readonly", t["surface_alt"])], foreground=[("readonly", t["text"])])
    style.configure("Treeview", background=t["surface"], fieldbackground=t["surface"], foreground=t["text"], rowheight=28)
    style.configure("Treeview.Heading", background=t["surface_alt"], foreground=t["text"], font=("Helvetica", 9, "bold"))
    theme_button.configure(text="☾  Dark mode" if current_theme == "light" else "☀  Light mode")


def toggle_theme():
    global current_theme
    current_theme = "dark" if current_theme == "light" else "light"
    apply_theme()
    status_var.set("The mood has changed. Make something beautiful. ✦")


def animate_header():
    global animation_phase
    animation_phase = (animation_phase + 1) % 4
    ornaments = ["✦     ✧", "✧     ✦", "✦     ✦", "✧     ✧"]
    eyebrow.configure(text=f"{ornaments[animation_phase]}  THE LANGUAGE JOURNAL  {ornaments[(animation_phase + 2) % 4]}")
    root.after(850, animate_header)


def animate_spinner():
    global spinner_phase
    if not translation_busy:
        return
    spinner_phase = (spinner_phase + 1) % 4
    dots = ["·", "··", "···", "··"]
    status_var.set("Translating your words " + dots[spinner_phase] + " ✦")
    root.after(350, animate_spinner)


# -------------------- BUILD THE INTERFACE --------------------
root = tk.Tk()
root.title(APP_TITLE)
root.geometry("900x790")
root.minsize(720, 680)
root.configure(bg=THEMES["light"]["bg"])

style = ttk.Style(root)
style.theme_use("clam")

# Top strip
# A restrained editorial layout with warm paper tones, wine accents, and magical glints.
topbar = tk.Frame(root, bg=THEMES["light"]["bg"])
topbar.pack(fill=tk.X, padx=28, pady=(18, 0))

eyebrow = tk.Label(topbar, text="✦     THE LANGUAGE JOURNAL     ✧", font=("Helvetica", 9, "bold"),
                   bg=THEMES["light"]["bg"], fg=THEMES["light"]["accent"])
eyebrow.pack(side=tk.LEFT)
theme_button = tk.Button(topbar, text="☾  Dark mode", command=toggle_theme, relief="flat", bd=0,
                         padx=13, pady=7, cursor="hand2", font=("Helvetica", 9, "bold"))
theme_button.pack(side=tk.RIGHT)

hero = tk.Frame(root, bg=THEMES["light"]["bg"])
hero.pack(fill=tk.X, padx=28, pady=(18, 16))
title_label = tk.Label(hero, text="WordGarden", font=("Georgia", 32, "bold"),
                       bg=THEMES["light"]["bg"], fg=THEMES["light"]["accent"])
title_label.pack(anchor="w")
subtitle_label = tk.Label(hero, text="A considered space for words, meaning, and little discoveries.",
                          font=("Georgia", 11, "italic"), bg=THEMES["light"]["bg"], fg=THEMES["light"]["muted"])
subtitle_label.pack(anchor="w", pady=(3, 0))

controls_card = tk.Frame(root, bg=THEMES["light"]["surface"], highlightthickness=1, highlightbackground=THEMES["light"]["line"])
controls_card.pack(fill=tk.X, padx=28, pady=(0, 14))
lang_inner = tk.Frame(controls_card, bg=THEMES["light"]["surface"])
lang_inner.pack(fill=tk.X, padx=18, pady=15)
section_from = tk.Label(lang_inner, text="01  FROM", font=("Helvetica", 9, "bold"), bg=THEMES["light"]["surface"], fg=THEMES["light"]["text"])
section_from.grid(row=0, column=0, sticky="w", padx=(0, 8), pady=(0, 7))
section_to = tk.Label(lang_inner, text="02  TRANSLATE TO", font=("Helvetica", 9, "bold"), bg=THEMES["light"]["surface"], fg=THEMES["light"]["text"])
section_to.grid(row=0, column=2, sticky="w", padx=(20, 8), pady=(0, 7))
source_lang_combo = ttk.Combobox(lang_inner, values=list(LANGUAGES.keys()), state="readonly", width=24, font=("Helvetica", 10))
source_lang_combo.set("English")
source_lang_combo.grid(row=1, column=0, sticky="ew")
arrow_label = tk.Label(lang_inner, text="→", font=("Georgia", 17), bg=THEMES["light"]["surface"], fg=THEMES["light"]["accent"])
arrow_label.grid(row=1, column=1, padx=14)
target_lang_combo = ttk.Combobox(lang_inner, values=list(LANGUAGES.keys()), state="readonly", width=24, font=("Helvetica", 10))
target_lang_combo.set("Hindi")
target_lang_combo.grid(row=1, column=2, sticky="ew")
lang_inner.columnconfigure(0, weight=1)
lang_inner.columnconfigure(2, weight=1)

workspace_card = tk.Frame(root, bg=THEMES["light"]["surface"], highlightthickness=1, highlightbackground=THEMES["light"]["line"])
workspace_card.pack(fill=tk.BOTH, expand=True, padx=28, pady=(0, 14))
workspace = tk.Frame(workspace_card, bg=THEMES["light"]["surface"])
workspace.pack(fill=tk.BOTH, expand=True, padx=18, pady=16)
input_label = tk.Label(workspace, text="03  YOUR WORDS", font=("Helvetica", 9, "bold"), bg=THEMES["light"]["surface"], fg=THEMES["light"]["text"])
input_label.pack(anchor="w", pady=(0, 6))
source_text = ScrolledText(workspace, height=6, wrap=tk.WORD, font=("Georgia", 11), relief="flat", bd=0,
                           padx=12, pady=10, undo=True, highlightthickness=1, highlightbackground=THEMES["light"]["line"])
source_text.pack(fill=tk.BOTH, expand=True, pady=(0, 14))
output_label = tk.Label(workspace, text="04  THE TRANSLATION", font=("Helvetica", 9, "bold"), bg=THEMES["light"]["surface"], fg=THEMES["light"]["text"])
output_label.pack(anchor="w", pady=(0, 6))
result_text = ScrolledText(workspace, height=6, wrap=tk.WORD, font=("Georgia", 11), relief="flat", bd=0,
                           padx=12, pady=10, highlightthickness=1, highlightbackground=THEMES["light"]["line"])
result_text.pack(fill=tk.BOTH, expand=True)

# Action row
# The buttons use text and contrast instead of images, so the app has no extra image dependencies.
actions_frame = tk.Frame(root, bg=THEMES["light"]["surface"])
actions_frame.pack(fill=tk.X, padx=28, pady=(0, 12))

def make_button(parent, text, command, primary=False):
    theme = THEMES[current_theme]
    return tk.Button(parent, text=text, command=command, relief="flat", bd=0,
                     padx=13, pady=10, cursor="hand2", font=("Helvetica", 9, "bold"),
                     bg=theme["accent"] if primary else theme["button_secondary"],
                     fg=theme["accent_text"] if primary else theme["button_secondary_text"],
                     activebackground=theme["accent_hover"] if primary else theme["line"],
                     activeforeground=theme["accent_text"] if primary else theme["text"])

btn_random = make_button(actions_frame, "✧  Word inspiration", load_random_word)
btn_random.pack(side=tk.LEFT, padx=(0, 7))
btn_translate = make_button(actions_frame, "✦  Translate", translate_text, primary=True)
btn_translate.pack(side=tk.LEFT, padx=(0, 7))
btn_save = make_button(actions_frame, "♡  Save word", save_favourite)
btn_save.pack(side=tk.LEFT, padx=(0, 7))
btn_favourites = make_button(actions_frame, "☷  Collection", show_favourites)
btn_favourites.pack(side=tk.LEFT, padx=(0, 7))
btn_clear = make_button(actions_frame, "Clear", clear_text)
btn_clear.pack(side=tk.RIGHT)

status_frame = tk.Frame(root, bg=THEMES["light"]["bg"])
status_frame.pack(fill=tk.X, padx=28, pady=(0, 6))
status_var = tk.StringVar(value="A fresh page awaits your words. ✦")
status_label = tk.Label(status_frame, textvariable=status_var, font=("Helvetica", 9),
                         bg=THEMES["light"]["bg"], fg=THEMES["light"]["muted"], anchor="w")
status_label.pack(side=tk.LEFT)
footer_label = tk.Label(status_frame, text="WORDS HAVE WORLDS INSIDE", font=("Helvetica", 8, "bold"),
                        bg=THEMES["light"]["bg"], fg=THEMES["light"]["muted"])
footer_label.pack(side=tk.RIGHT)

# Theme styling and a small, continuous sparkle animation.
apply_theme()
animate_header()
root.mainloop()
